In [34]:
import sys
import json
import joblib
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import sklearn
import mlflow
import mlflow.sklearn

warnings.filterwarnings("ignore")

print("Python version:", sys.version.split()[0])
print("pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("scikit-learn version:", sklearn.__version__)
print("MLflow version:", mlflow.__version__)
print("Libraries imported successfully.")

Python version: 3.14.2
pandas version: 3.0.5
NumPy version: 2.5.2
scikit-learn version: 1.9.0
MLflow version: 3.16.1
Libraries imported successfully.


2. Define Project Path

In [35]:
project_folder = Path.cwd()

model_file = project_folder / "final_production_candidate.joblib"
mlflow_database = project_folder / "nordex_mlflow.db"

possible_data_files = [
    project_folder / "final_feature_set.csv",
    project_folder / "final_modeling_dataset.csv"
]

data_file = next(
    (file for file in possible_data_files if file.exists()),
    None
)

file_check = pd.DataFrame({
    "Required item": [
        "Project folder",
        "Production model",
        "MLflow database",
        "Modeling dataset"
    ],
    "Location or status": [
        str(project_folder),
        model_file.exists(),
        mlflow_database.exists(),
        str(data_file) if data_file else "Not found"
    ]
})

file_check

,Required item,Location or status
0,Project folder,c:\Users\User\AMDARI Project 1
1,Production model,True
2,MLflow database,True
3,Modeling dataset,c:\Users\User\AMDARI Project 1\final_feature_s...


3. Load the Selected Model

In [36]:
manufacturing_data = pd.read_csv(data_file)

target_column = "shift_efficiency_score"

print("Dataset loaded successfully.")
print("Dataset shape:", manufacturing_data.shape)
print("Target column present:", target_column in manufacturing_data.columns)

display(manufacturing_data.head())

Dataset loaded successfully.
Dataset shape: (20, 9)
Target column present: False


,Feature,absolute_correlation,Random Forest importance,rfe_selected,best_rfe_ranking,correlation_score,importance_score,combined_score,Final selection
0,production_rate_per_hour,0.922031,0.741228,1,1,0.975256,1.000000,0.992577,Selected
1,good_units,0.926939,0.069162,1,1,0.980448,0.093307,0.540788,Selected
2,units_produced,0.927178,0.022919,1,1,0.980701,0.030920,0.509670,Selected
3,shift_start_hour,0.945424,0.010380,1,1,1.000000,0.014003,0.507002,Selected
4,estimated_oee_pct,0.926716,0.018857,1,1,0.980211,0.025441,0.506784,Selected


Load The Selected Production Model


In [37]:
if not model_file.exists():
    raise FileNotFoundError(
        f"Production model was not found: {model_file}"
    )

local_model = joblib.load(model_file)

print("Production model loaded successfully.")
print("Object type:", type(local_model).__name__)

if hasattr(local_model, "named_steps"):
    print("Pipeline steps:")
    
    for step_name, step_object in local_model.named_steps.items():
        print(
            f" - {step_name}: "
            f"{type(step_object).__name__}"
        )

Production model loaded successfully.
Object type: Pipeline
Pipeline steps:
 - preprocessor: ColumnTransformer
 - model: RandomForestRegressor


In [38]:
if hasattr(local_model, "feature_names_in_"):
    expected_columns = list(
        local_model.feature_names_in_
    )
else:
    expected_columns = [
        column
        for column in manufacturing_data.columns
        if column != target_column
    ]

missing_from_dataset = [
    column
    for column in expected_columns
    if column not in manufacturing_data.columns
]

if missing_from_dataset:
    raise ValueError(
        "The dataset is missing model features: "
        f"{missing_from_dataset}"
    )

reference_features = manufacturing_data[
    expected_columns
].copy()

numerical_columns = reference_features.select_dtypes(
    include=np.number
).columns.tolist()

categorical_columns = [
    column
    for column in expected_columns
    if column not in numerical_columns
]

schema_summary = pd.DataFrame({
    "Measure": [
        "Expected predictors",
        "Numerical predictors",
        "Categorical predictors",
        "Reference rows",
        "Missing expected columns"
    ],
    "Result": [
        len(expected_columns),
        len(numerical_columns),
        len(categorical_columns),
        len(reference_features),
        len(missing_from_dataset)
    ]
})

display(schema_summary)

print("\nNumerical columns:")
print(numerical_columns)

print("\nCategorical columns:")
print(categorical_columns)

ValueError: The dataset is missing model features: ['production_rate_per_hour', 'good_units', 'units_produced', 'shift_start_hour', 'estimated_oee_pct', 'downtime_minutes', 'runtime_hours', 'defect_rate_pct', 'experience_runtime_interaction', 'temperature_deviation', 'defect_count', 'shift_name', 'experience_level', 'cycle_time_avg', 'machine_prior_avg_downtime', 'machine_prior_avg_defect_rate', 'machine_id', 'machine_prior_avg_units', 'maintenance_downtime', 'humidity']

In [39]:
csv_files = list(project_folder.glob("*.csv"))

csv_inventory = []

for csv_file in csv_files:
    try:
        temporary_data = pd.read_csv(csv_file)

        csv_inventory.append({
            "File": csv_file.name,
            "Rows": temporary_data.shape[0],
            "Columns": temporary_data.shape[1],
            "First columns": ", ".join(
                temporary_data.columns[:5]
            )
        })

    except Exception as error:
        csv_inventory.append({
            "File": csv_file.name,
            "Rows": "Could not read",
            "Columns": "Could not read",
            "First columns": str(error)
        })

csv_inventory = pd.DataFrame(csv_inventory)

display(csv_inventory)

,File,Rows,Columns,First columns
0,cleaned_manufacturing_dataset_v1.csv,10117,41,"shift_id, shift_name, start_time, end_time, su..."
1,clean_manufacturing_dataset_v1.csv,10117,41,"shift_id, shift_name, start_time, end_time, su..."
2,day5_cross_validation_results.csv,5,5,"Model, CV MAE, CV RMSE, CV R-squared, RMSE sta..."
3,day5_feature_importance.csv,71,2,"Feature, Importance"
4,day5_final_model_comparison.csv,6,5,"Model, MAE, RMSE, R-squared, MLflow run ID"
5,day5_final_model_selection.csv,7,2,"Selection measure, Result"
6,extracted dataset file.csv,10117,31,"shift_id, shift_name, start_time, end_time, su..."
7,feature_engineered_manufacturing_dataset.csv,10117,67,"shift_id, shift_name, start_time, end_time, su..."
8,feature_engineering_documentation.csv,26,4,"Feature, Category, Transformation, Reason"
9,feature_selection_comparison.csv,31,8,"Feature, absolute_correlation, Random Forest i..."


In [40]:
modeling_data_file = (
    project_folder
    / "final_modeling_dataset.csv"
)

if not modeling_data_file.exists():
    raise FileNotFoundError(
        "final_modeling_dataset.csv was not found "
        "inside the project folder."
    )

manufacturing_data = pd.read_csv(
    modeling_data_file
)

data_file = modeling_data_file

print("Correct dataset loaded.")
print("File:", data_file.name)
print("Shape:", manufacturing_data.shape)
print("Number of columns:", manufacturing_data.shape[1])

print("\nDataset columns:")
print(manufacturing_data.columns.tolist())

Correct dataset loaded.
File: final_modeling_dataset.csv
Shape: (10117, 21)
Number of columns: 21

Dataset columns:
['production_rate_per_hour', 'good_units', 'units_produced', 'shift_start_hour', 'estimated_oee_pct', 'downtime_minutes', 'runtime_hours', 'defect_rate_pct', 'experience_runtime_interaction', 'temperature_deviation', 'defect_count', 'shift_name', 'experience_level', 'cycle_time_avg', 'machine_prior_avg_downtime', 'machine_prior_avg_defect_rate', 'machine_id', 'machine_prior_avg_units', 'maintenance_downtime', 'humidity', 'shift_efficiency_score']


In [41]:
target_column = "shift_efficiency_score"

if hasattr(local_model, "feature_names_in_"):
    expected_columns = list(
        local_model.feature_names_in_
    )
else:
    expected_columns = [
        column
        for column in manufacturing_data.columns
        if column != target_column
    ]

missing_from_dataset = [
    column
    for column in expected_columns
    if column not in manufacturing_data.columns
]

extra_dataset_columns = [
    column
    for column in manufacturing_data.columns
    if (
        column not in expected_columns
        and column != target_column
    )
]

schema_diagnosis = pd.DataFrame({
    "Schema measure": [
        "Model predictors required",
        "Dataset columns available",
        "Missing model predictors",
        "Additional dataset columns",
        "Target column available"
    ],
    "Result": [
        len(expected_columns),
        manufacturing_data.shape[1],
        len(missing_from_dataset),
        len(extra_dataset_columns),
        target_column in manufacturing_data.columns
    ]
})

display(schema_diagnosis)

if missing_from_dataset:
    print("\nMissing columns:")
    print(missing_from_dataset)

else:
    reference_features = manufacturing_data[
        expected_columns
    ].copy()

    numerical_columns = (
        reference_features.select_dtypes(
            include=np.number
        ).columns.tolist()
    )

    categorical_columns = [
        column
        for column in expected_columns
        if column not in numerical_columns
    ]

    print("\nInput schema created successfully.")
    print(
        "Expected predictors:",
        len(expected_columns)
    )
    print(
        "Numerical predictors:",
        len(numerical_columns)
    )
    print(
        "Categorical predictors:",
        len(categorical_columns)
    )

,Schema measure,Result
0,Model predictors required,20
1,Dataset columns available,21
2,Missing model predictors,0
3,Additional dataset columns,0
4,Target column available,True



Input schema created successfully.
Expected predictors: 20
Numerical predictors: 18
Categorical predictors: 2


In [42]:
pipeline_module_code = '''
from datetime import datetime, timezone

import numpy as np
import pandas as pd


class ManufacturingPredictionPipeline:
    """Validate manufacturing records and generate efficiency predictions."""

    def __init__(
        self,
        model,
        expected_columns,
        numerical_columns,
        categorical_columns
    ):
        self.model = model
        self.expected_columns = list(expected_columns)
        self.numerical_columns = list(numerical_columns)
        self.categorical_columns = list(categorical_columns)

    def validate(self, data):
        if isinstance(data, dict):
            data = pd.DataFrame([data])
        elif not isinstance(data, pd.DataFrame):
            raise TypeError(
                "Input must be a pandas DataFrame or dictionary."
            )

        if data.empty:
            raise ValueError("Input data contains no rows.")

        missing_columns = [
            column
            for column in self.expected_columns
            if column not in data.columns
        ]

        if missing_columns:
            raise ValueError(
                "Missing required columns: "
                + ", ".join(missing_columns)
            )

        validated_data = data[
            self.expected_columns
        ].copy()

        for column in self.numerical_columns:
            original_non_null = validated_data[column].notna()

            converted = pd.to_numeric(
                validated_data[column],
                errors="coerce"
            )

            invalid_values = (
                original_non_null
                & converted.isna()
            ).sum()

            if invalid_values > 0:
                raise ValueError(
                    f"Column '{column}' contains "
                    f"{invalid_values} non-numeric value(s)."
                )

            validated_data[column] = converted

        for column in self.categorical_columns:
            validated_data[column] = (
                validated_data[column]
                .astype("string")
            )

        non_negative_columns = [
            "units_produced",
            "good_units",
            "defect_count",
            "runtime_hours",
            "downtime_minutes",
            "cycle_time_avg",
            "production_rate_per_hour"
        ]

        for column in non_negative_columns:
            if (
                column in validated_data.columns
                and (validated_data[column].dropna() < 0).any()
            ):
                raise ValueError(
                    f"Column '{column}' cannot contain "
                    "negative values."
                )

        if {
            "good_units",
            "units_produced"
        }.issubset(validated_data.columns):
            invalid_good_units = (
                validated_data["good_units"]
                > validated_data["units_produced"]
            )

            if invalid_good_units.any():
                raise ValueError(
                    "good_units cannot exceed units_produced."
                )

        if "humidity" in validated_data.columns:
            invalid_humidity = ~validated_data[
                "humidity"
            ].dropna().between(0, 100)

            if invalid_humidity.any():
                raise ValueError(
                    "humidity must be between 0 and 100."
                )

        return validated_data

    def predict(self, data):
        validated_data = self.validate(data)

        predictions = self.model.predict(
            validated_data
        )

        results = pd.DataFrame({
            "predicted_shift_efficiency_score":
                np.asarray(predictions, dtype=float)
        })

        results["prediction_timestamp_utc"] = (
            datetime.now(timezone.utc).isoformat()
        )

        results["pipeline_status"] = "Successful"

        return results
'''

module_file = project_folder / "production_pipeline.py"
module_file.write_text(
    pipeline_module_code,
    encoding="utf-8"
)

print("Pipeline module created:", module_file.name)
print("File exists:", module_file.exists())

Pipeline module created: production_pipeline.py
File exists: True


In [43]:
import importlib
import production_pipeline

importlib.reload(production_pipeline)

ManufacturingPredictionPipeline = (
    production_pipeline.ManufacturingPredictionPipeline
)

production_pipeline = ManufacturingPredictionPipeline(
    model=local_model,
    expected_columns=expected_columns,
    numerical_columns=numerical_columns,
    categorical_columns=categorical_columns
)

print("Production pipeline created successfully.")
print(
    "Required predictor count:",
    len(production_pipeline.expected_columns)
)
print(
    "Numerical predictor count:",
    len(production_pipeline.numerical_columns)
)
print(
    "Categorical predictor count:",
    len(production_pipeline.categorical_columns)
)

Production pipeline created successfully.
Required predictor count: 20
Numerical predictor count: 18
Categorical predictor count: 2


In [44]:
sample_input = reference_features.head(5).copy()

validation_result = production_pipeline.validate(
    sample_input
)

sample_predictions = production_pipeline.predict(
    sample_input
)

prediction_test_results = pd.concat(
    [
        sample_input.reset_index(drop=True),
        sample_predictions
    ],
    axis=1
)

print("Input validation successful.")
print("Rows tested:", len(sample_input))
print(
    "Predictions generated:",
    len(sample_predictions)
)

display(
    prediction_test_results[
        [
            "predicted_shift_efficiency_score",
            "pipeline_status"
        ]
    ]
)

Input validation successful.
Rows tested: 5
Predictions generated: 5


,predicted_shift_efficiency_score,pipeline_status
0,95.747173,Successful
1,95.747173,Successful
2,95.747173,Successful
3,95.747173,Successful
4,95.747173,Successful


In [45]:
missing_column_test = sample_input.drop(
    columns=[expected_columns[0]]
)

try:
    production_pipeline.predict(
        missing_column_test
    )

except ValueError as error:
    print("Validation test passed.")
    print("Rejected invalid input:")
    print(error)

Validation test passed.
Rejected invalid input:
Missing required columns: production_rate_per_hour


In [46]:
invalid_numeric_test = sample_input.copy()

test_numeric_column = numerical_columns[0]

# Convert this column to an object type so the test can
# deliberately insert an invalid text value.
invalid_numeric_test[test_numeric_column] = (
    invalid_numeric_test[test_numeric_column]
    .astype("object")
)

invalid_numeric_test.at[
    invalid_numeric_test.index[0],
    test_numeric_column
] = "not-a-number"

print(
    "Testing invalid value in column:",
    test_numeric_column
)

try:
    production_pipeline.predict(
        invalid_numeric_test
    )

    print(
        "Test failed: invalid input was not rejected."
    )

except ValueError as error:
    print("Numeric validation test passed.")
    print("Rejected invalid input:")
    print(error)

Testing invalid value in column: production_rate_per_hour
Numeric validation test passed.
Rejected invalid input:
Column 'production_rate_per_hour' contains 1 non-numeric value(s).


In [47]:
unseen_category_test = sample_input.copy()

if categorical_columns:
    test_categorical_column = categorical_columns[0]

    unseen_category_test.loc[
        unseen_category_test.index[0],
        test_categorical_column
    ] = "Previously_Unseen_Category"

    unseen_predictions = production_pipeline.predict(
        unseen_category_test
    )

    print("Unseen-category test passed.")
    print(
        "Tested column:",
        test_categorical_column
    )

    display(unseen_predictions)

else:
    print(
        "No categorical predictors were detected, "
        "so this test was not required."
    )

Unseen-category test passed.
Tested column: shift_name


,predicted_shift_efficiency_score,prediction_timestamp_utc,pipeline_status
0,94.730059,2026-09-21T09:03:29.153933+00:00,Successful
1,95.747173,2026-09-21T09:03:29.153933+00:00,Successful
2,95.747173,2026-09-21T09:03:29.153933+00:00,Successful
3,95.747173,2026-09-21T09:03:29.153933+00:00,Successful
4,95.747173,2026-09-21T09:03:29.153933+00:00,Successful


In [48]:
tracking_uri = (
    "sqlite:///"
    + mlflow_database.resolve().as_posix()
)

mlflow.set_tracking_uri(tracking_uri)

registered_model_name = (
    "NorDex_Shift_Efficiency_Model"
)

model_alias = "candidate"

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Registered model:", registered_model_name)
print("Requested alias:", model_alias)

MLflow tracking URI: sqlite:///C:/Users/User/AMDARI Project 1/nordex_mlflow.db
Registered model: NorDex_Shift_Efficiency_Model
Requested alias: candidate


In [49]:
from mlflow.tracking import MlflowClient

mlflow_client = MlflowClient()

candidate_version = (
    mlflow_client.get_model_version_by_alias(
        registered_model_name,
        model_alias
    )
)

registry_summary = pd.DataFrame({
    "Registry measure": [
        "Registered model",
        "Alias",
        "Model version",
        "Run ID",
        "Model status"
    ],
    "Result": [
        registered_model_name,
        model_alias,
        candidate_version.version,
        candidate_version.run_id,
        candidate_version.status
    ]
})

display(registry_summary)

,Registry measure,Result
0,Registered model,NorDex_Shift_Efficiency_Model
1,Alias,candidate
2,Model version,2
3,Run ID,0f824710fb39487eb35f583f4c35ecb1
4,Model status,READY


In [50]:
mlflow_model_uri = (
    f"models:/{registered_model_name}"
    f"@{model_alias}"
)

try:
    registry_model = (
        mlflow.sklearn.load_model(
            mlflow_model_uri
        )
    )

    registry_source = "MLflow Model Registry"

    print(
        "Registered model loaded successfully."
    )
    print("Model URI:", mlflow_model_uri)
    print(
        "Loaded object type:",
        type(registry_model).__name__
    )

except Exception as error:
    registry_model = local_model
    registry_source = (
        "Local production artifact fallback"
    )

    print(
        "MLflow model loading was unavailable."
    )
    print("Reason:", str(error))
    print(
        "The verified local production model "
        "was loaded instead."
    )

Registered model loaded successfully.
Model URI: models:/NorDex_Shift_Efficiency_Model@candidate
Loaded object type: Pipeline


In [51]:
registry_production_pipeline = (
    ManufacturingPredictionPipeline(
        model=registry_model,
        expected_columns=expected_columns,
        numerical_columns=numerical_columns,
        categorical_columns=categorical_columns
    )
)

registry_predictions = (
    registry_production_pipeline.predict(
        sample_input
    )
)

prediction_comparison = pd.DataFrame({
    "Local model prediction":
        production_pipeline.predict(
            sample_input
        )[
            "predicted_shift_efficiency_score"
        ],
    "Registry model prediction":
        registry_predictions[
            "predicted_shift_efficiency_score"
        ]
})

prediction_comparison[
    "Absolute difference"
] = (
    prediction_comparison[
        "Local model prediction"
    ]
    - prediction_comparison[
        "Registry model prediction"
    ]
).abs()

display(prediction_comparison)

print(
    "Maximum prediction difference:",
    prediction_comparison[
        "Absolute difference"
    ].max()
)

,Local model prediction,Registry model prediction,Absolute difference
0,95.747173,95.747173,0.0
1,95.747173,95.747173,0.0
2,95.747173,95.747173,0.0
3,95.747173,95.747173,0.0
4,95.747173,95.747173,0.0


Maximum prediction difference: 0.0


In [52]:
version_number = str(
    candidate_version.version
)

versioned_pipeline_file = (
    project_folder
    / f"nordex_production_pipeline_v{version_number}.joblib"
)

joblib.dump(
    registry_production_pipeline,
    versioned_pipeline_file
)

print(
    "Versioned pipeline:",
    versioned_pipeline_file.name
)
print(
    "Pipeline saved:",
    versioned_pipeline_file.exists()
)
print(
    "Pipeline source:",
    registry_source
)

Versioned pipeline: nordex_production_pipeline_v2.joblib
Pipeline saved: True
Pipeline source: MLflow Model Registry


In [53]:
reloaded_pipeline = joblib.load(
    versioned_pipeline_file
)

reloaded_predictions = reloaded_pipeline.predict(
    sample_input
)

reload_check = np.allclose(
    registry_predictions[
        "predicted_shift_efficiency_score"
    ].to_numpy(),
    reloaded_predictions[
        "predicted_shift_efficiency_score"
    ].to_numpy()
)

print("Saved pipeline reloaded successfully.")
print("Predictions match:", reload_check)

display(reloaded_predictions)

Saved pipeline reloaded successfully.
Predictions match: True


,predicted_shift_efficiency_score,prediction_timestamp_utc,pipeline_status
0,95.747173,2026-09-21T09:03:52.170113+00:00,Successful
1,95.747173,2026-09-21T09:03:52.170113+00:00,Successful
2,95.747173,2026-09-21T09:03:52.170113+00:00,Successful
3,95.747173,2026-09-21T09:03:52.170113+00:00,Successful
4,95.747173,2026-09-21T09:03:52.170113+00:00,Successful


In [54]:
pipeline_configuration = {
    "project": (
        "NorDex Manufacturing Shift "
        "Efficiency Prediction"
    ),
    "pipeline_version": version_number,
    "registered_model": registered_model_name,
    "model_alias": model_alias,
    "model_source": registry_source,
    "target": target_column,
    "expected_predictors": expected_columns,
    "numerical_predictors": numerical_columns,
    "categorical_predictors": categorical_columns,
    "preprocessing": (
        "Stored inside the fitted "
        "scikit-learn pipeline"
    ),
    "unknown_category_handling": "ignore",
    "validation_enabled": True,
    "created_utc": (
        datetime.utcnow().isoformat()
        + "Z"
    )
}

configuration_file = (
    project_folder
    / "production_pipeline_config.json"
)

with open(
    configuration_file,
    "w",
    encoding="utf-8"
) as configuration_output:
    json.dump(
        pipeline_configuration,
        configuration_output,
        indent=4
    )

print(
    "Configuration file:",
    configuration_file.name
)
print(
    "Configuration saved:",
    configuration_file.exists()
)

display(
    pd.DataFrame({
        "Configuration item": [
            "Pipeline version",
            "Registered model",
            "Alias",
            "Predictor count",
            "Validation enabled"
        ],
        "Value": [
            version_number,
            registered_model_name,
            model_alias,
            len(expected_columns),
            True
        ]
    })
)

Configuration file: production_pipeline_config.json
Configuration saved: True


,Configuration item,Value
0,Pipeline version,2
1,Registered model,NorDex_Shift_Efficiency_Model
2,Alias,candidate
3,Predictor count,20
4,Validation enabled,True


In [55]:
prediction_report_file = (
    project_folder
    / "day1_prediction_testing_report.csv"
)

prediction_test_results.to_csv(
    prediction_report_file,
    index=False
)

validation_test_report = pd.DataFrame({
    "Test": [
        "Valid manufacturing records",
        "Missing required column",
        "Non-numeric predictor",
        "Unseen categorical value",
        "Saved pipeline reload"
    ],
    "Expected behaviour": [
        "Generate predictions",
        "Reject input",
        "Reject input",
        "Generate prediction safely",
        "Reproduce original predictions"
    ],
    "Result": [
        "Passed",
        "Passed",
        "Passed",
        (
            "Passed"
            if categorical_columns
            else "Not applicable"
        ),
        "Passed" if reload_check else "Failed"
    ]
})

validation_report_file = (
    project_folder
    / "day1_validation_test_report.csv"
)

validation_test_report.to_csv(
    validation_report_file,
    index=False
)

display(validation_test_report)

,Test,Expected behaviour,Result
0,Valid manufacturing records,Generate predictions,Passed
1,Missing required column,Reject input,Passed
2,Non-numeric predictor,Reject input,Passed
3,Unseen categorical value,Generate prediction safely,Passed
4,Saved pipeline reload,Reproduce original predictions,Passed


In [56]:
day1_deliverables = [
    module_file,
    configuration_file,
    versioned_pipeline_file,
    prediction_report_file,
    validation_report_file
]

deliverable_check = pd.DataFrame({
    "Deliverable": [
        file.name
        for file in day1_deliverables
    ],
    "Exists": [
        file.exists()
        for file in day1_deliverables
    ],
    "Size in KB": [
        round(
            file.stat().st_size / 1024,
            2
        )
        if file.exists()
        else 0
        for file in day1_deliverables
    ]
})

display(deliverable_check)

if deliverable_check["Exists"].all():
    print(
        "Week 3 Day 1 pipeline completed successfully."
    )
else:
    print(
        "One or more deliverables are missing."
    )

,Deliverable,Exists,Size in KB
0,production_pipeline.py,True,3.95
1,production_pipeline_config.json,True,1.76
2,nordex_production_pipeline_v2.joblib,True,3583.91
3,day1_prediction_testing_report.csv,True,1.68
4,day1_validation_test_report.csv,True,0.29


Week 3 Day 1 pipeline completed successfully.


In [57]:
manufacturing_data = pd.read_csv(
    data_file
)

local_model = joblib.load(
    model_file
)

print(
    "Dataset shape:",
    manufacturing_data.shape
)

print(
    "Object type:",
    type(local_model).__name__
)

print([
    (
        name,
        type(step).__name__
    )
    for name, step in local_model.steps
])

Dataset shape: (10117, 21)
Object type: Pipeline
[('preprocessor', 'ColumnTransformer'), ('model', 'RandomForestRegressor')]


In [58]:
production_pipeline = (
    ManufacturingPredictionPipeline(
        model=local_model,
        expected_columns=expected_columns,
        numerical_columns=numerical_columns,
        categorical_columns=categorical_columns
    )
)

print(
    "Required predictor count:",
    len(expected_columns)
)

print(
    "Numerical predictor count:",
    len(numerical_columns)
)

print(
    "Categorical predictor count:",
    len(categorical_columns)
)

Required predictor count: 20
Numerical predictor count: 18
Categorical predictor count: 2


In [59]:
expected_columns = list(
    local_model.feature_names_in_
)

reference_features = manufacturing_data[
    expected_columns
].copy()

numerical_columns = (
    reference_features.select_dtypes(
        include=np.number
    ).columns.tolist()
)

categorical_columns = [
    column
    for column in expected_columns
    if column not in numerical_columns
]

print(
    "Expected predictors:",
    len(expected_columns)
)

print(
    "Numerical predictors:",
    len(numerical_columns)
)

print(
    "Categorical predictors:",
    categorical_columns
)

Expected predictors: 20
Numerical predictors: 18
Categorical predictors: ['shift_name', 'machine_id']


In [60]:
unseen_input = sample_input.copy()

unseen_input.at[
    unseen_input.index[0],
    "shift_name"
] = "Previously_Unseen_Category"

unseen_predictions = (
    production_pipeline.predict(
        unseen_input
    )
)

print(
    "Unseen-category test passed."
)

Unseen-category test passed.


In [61]:
sample_input = (
    reference_features
    .head(5)
    .copy()
)

sample_predictions = (
    production_pipeline.predict(
        sample_input
    )
)

prediction_test_results = pd.concat(
    [
        sample_input.reset_index(
            drop=True
        ),
        sample_predictions.reset_index(
            drop=True
        )
    ],
    axis=1
)

display(
    prediction_test_results[
        [
            "predicted_shift_efficiency_score",
            "pipeline_status"
        ]
    ]
)

,predicted_shift_efficiency_score,pipeline_status
0,95.747173,Successful
1,95.747173,Successful
2,95.747173,Successful
3,95.747173,Successful
4,95.747173,Successful


In [62]:
missing_column_test = (
    sample_input.drop(
        columns=[
            "production_rate_per_hour"
        ]
    )
)

try:
    production_pipeline.predict(
        missing_column_test
    )

except ValueError as error:
    print(error)

Missing required columns: production_rate_per_hour


In [63]:
invalid_numeric_test = (
    sample_input.copy()
)

invalid_numeric_test[
    "production_rate_per_hour"
] = invalid_numeric_test[
    "production_rate_per_hour"
].astype(object)

invalid_numeric_test.at[
    invalid_numeric_test.index[0],
    "production_rate_per_hour"
] = "not-a-number"

try:
    production_pipeline.predict(
        invalid_numeric_test
    )

except ValueError as error:
    print(error)

Column 'production_rate_per_hour' contains 1 non-numeric value(s).


In [64]:
from pathlib import Path
import mlflow
import mlflow.sklearn

project_folder = Path.cwd()

mlflow_database = (
    project_folder
    / "nordex_mlflow.db"
)

tracking_uri = (
    "sqlite:///"
    + mlflow_database.resolve().as_posix()
)

mlflow.set_tracking_uri(
    tracking_uri
)

registered_model_name = (
    "NorDex_Shift_Efficiency_Model"
)

model_alias = "candidate"

print(
    "MLflow tracking URI:",
    mlflow.get_tracking_uri()
)

MLflow tracking URI: sqlite:///C:/Users/User/AMDARI Project 1/nordex_mlflow.db


In [65]:
from mlflow.tracking import MlflowClient

mlflow_client = MlflowClient()

candidate_version = (
    mlflow_client
    .get_model_version_by_alias(
        registered_model_name,
        model_alias
    )
)

print(
    "Registered model:",
    candidate_version.name
)

print(
    "Candidate version:",
    candidate_version.version
)

print(
    "Status:",
    candidate_version.status
)

print(
    "Run ID:",
    candidate_version.run_id
)

Registered model: NorDex_Shift_Efficiency_Model
Candidate version: 2
Status: READY
Run ID: 0f824710fb39487eb35f583f4c35ecb1


In [66]:
from mlflow.tracking import MlflowClient

mlflow_client = MlflowClient()

candidate_version = (
    mlflow_client
    .get_model_version_by_alias(
        registered_model_name,
        model_alias
    )
)

print(
    "Registered model:",
    candidate_version.name
)

print(
    "Candidate version:",
    candidate_version.version
)

print(
    "Status:",
    candidate_version.status
)

print(
    "Run ID:",
    candidate_version.run_id
)

Registered model: NorDex_Shift_Efficiency_Model
Candidate version: 2
Status: READY
Run ID: 0f824710fb39487eb35f583f4c35ecb1


In [67]:
mlflow_model_uri = (
    "models:/"
    "NorDex_Shift_Efficiency_Model"
    "@candidate"
)

registry_model = (
    mlflow.sklearn.load_model(
        mlflow_model_uri
    )
)

print(
    "Registered model loaded successfully."
)

print(
    "Model URI:",
    mlflow_model_uri
)

print(
    "Loaded object type:",
    type(registry_model).__name__
)

Registered model loaded successfully.
Model URI: models:/NorDex_Shift_Efficiency_Model@candidate
Loaded object type: Pipeline


In [68]:
registry_production_pipeline = (
    ManufacturingPredictionPipeline(
        model=registry_model,
        expected_columns=expected_columns,
        numerical_columns=numerical_columns,
        categorical_columns=categorical_columns
    )
)

local_values = production_pipeline.predict(
    sample_input
)[
    "predicted_shift_efficiency_score"
].to_numpy()

registry_values = (
    registry_production_pipeline
    .predict(sample_input)[
        "predicted_shift_efficiency_score"
    ].to_numpy()
)

maximum_difference = np.max(
    np.abs(
        local_values
        - registry_values
    )
)

print(
    "Maximum prediction difference:",
    maximum_difference
)

Maximum prediction difference: 0.0


In [69]:
version_number = str(
    candidate_version.version
)

versioned_pipeline_file = (
    project_folder
    / (
        "nordex_production_pipeline_v"
        + version_number
        + ".joblib"
    )
)

joblib.dump(
    registry_production_pipeline,
    versioned_pipeline_file
)

reloaded_pipeline = joblib.load(
    versioned_pipeline_file
)

original_values = (
    registry_production_pipeline
    .predict(sample_input)[
        "predicted_shift_efficiency_score"
    ].to_numpy()
)

reloaded_values = (
    reloaded_pipeline
    .predict(sample_input)[
        "predicted_shift_efficiency_score"
    ].to_numpy()
)

reload_check = np.allclose(
    original_values,
    reloaded_values
)

print(
    "Versioned pipeline:",
    versioned_pipeline_file.name
)

print(
    "Pipeline saved:",
    versioned_pipeline_file.exists()
)

print(
    "Predictions match:",
    reload_check
)

Versioned pipeline: nordex_production_pipeline_v2.joblib
Pipeline saved: True
Predictions match: True


In [71]:
from datetime import datetime, timezone
import json

pipeline_configuration = {
    "project": (
        "NorDex Manufacturing Shift "
        "Efficiency Prediction"
    ),
    "pipeline_version": version_number,
    "registered_model":
        registered_model_name,
    "model_alias": model_alias,
    "model_source":
        "MLflow Model Registry",
    "target": target_column,
    "expected_predictors":
        expected_columns,
    "numerical_predictors":
        numerical_columns,
    "categorical_predictors":
        categorical_columns,
    "preprocessing": (
        "Stored inside the fitted "
        "scikit-learn pipeline"
    ),
    "unknown_category_handling":
        "ignore",
    "validation_enabled": True,
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat()
}

configuration_file = (
    project_folder
    / "production_pipeline_config.json"
)

with open(
    configuration_file,
    "w",
    encoding="utf-8"
) as configuration_output:
    json.dump(
        pipeline_configuration,
        configuration_output,
        indent=4
    )

print(
    "Configuration file:",
    configuration_file.name
)

print(
    "Configuration saved:",
    configuration_file.exists()
)

Configuration file: production_pipeline_config.json
Configuration saved: True


In [72]:
prediction_report_file = (
    project_folder
    / "day1_prediction_testing_report.csv"
)

prediction_test_results.to_csv(
    prediction_report_file,
    index=False
)

validation_report_file = (
    project_folder
    / "day1_validation_test_report.csv"
)

validation_test_report.to_csv(
    validation_report_file,
    index=False
)

print(
    "Prediction report saved:",
    prediction_report_file.exists()
)

print(
    "Validation report saved:",
    validation_report_file.exists()
)

Prediction report saved: True
Validation report saved: True


In [73]:
module_file = (
    project_folder
    / "production_pipeline.py"
)

configuration_file = (
    project_folder
    / "production_pipeline_config.json"
)

versioned_pipeline_file = (
    project_folder
    / "nordex_production_pipeline_v2.joblib"
)

prediction_report_file = (
    project_folder
    / "day1_prediction_testing_report.csv"
)

validation_report_file = (
    project_folder
    / "day1_validation_test_report.csv"
)

day1_deliverables = [
    module_file,
    configuration_file,
    versioned_pipeline_file,
    prediction_report_file,
    validation_report_file
]

deliverable_check = pd.DataFrame({
    "Deliverable": [
        file.name
        for file in day1_deliverables
    ],
    "Exists": [
        file.exists()
        for file in day1_deliverables
    ],
    "Size in KB": [
        round(
            file.stat().st_size / 1024,
            2
        )
        if file.exists()
        else 0
        for file in day1_deliverables
    ]
})

display(deliverable_check)

print(
    "Completed deliverables:",
    f"{deliverable_check['Exists'].sum()}/"
    f"{len(deliverable_check)}"
)

if deliverable_check["Exists"].all():
    print(
        "Week 3 Day 1 pipeline "
        "completed successfully."
    )

,Deliverable,Exists,Size in KB
0,production_pipeline.py,True,3.95
1,production_pipeline_config.json,True,1.76
2,nordex_production_pipeline_v2.joblib,True,3583.91
3,day1_prediction_testing_report.csv,True,1.68
4,day1_validation_test_report.csv,True,0.29


Completed deliverables: 5/5
Week 3 Day 1 pipeline completed successfully.


In [74]:
import pandas as pd

expected_columns = list(
    local_model.feature_names_in_
)

predictor_schema = pd.DataFrame({
    "No.": range(1, len(expected_columns) + 1),
    "Predictor": expected_columns,
    "Type": [
        "Numerical"
        if pd.api.types.is_numeric_dtype(
            manufacturing_data[column]
        )
        else "Categorical"
        for column in expected_columns
    ]
})

display(predictor_schema)

,No.,Predictor,Type
0,1,production_rate_per_hour,Numerical
1,2,good_units,Numerical
2,3,units_produced,Numerical
3,4,shift_start_hour,Numerical
4,5,estimated_oee_pct,Numerical
5,6,downtime_minutes,Numerical
6,7,runtime_hours,Numerical
7,8,defect_rate_pct,Numerical
8,9,experience_runtime_interaction,Numerical
9,10,temperature_deviation,Numerical
